# Controlled Overfitting Data Selection for RSD

This notebook modernizes the 2019 "gold standard" data-selection idea for the current BariatricRSD project.

The old project used controlled per-video overfitting to identify frames whose RSD labels were internally consistent. This notebook keeps the idea but avoids depending on the old proprietary modules. It is designed to be run on the current MultiBypass140 manifest and to produce a filtered training manifest for follow-up experiments.

Core rule:

1. Split videos first.
2. Run the overfitting selector only on training videos.
3. For each training video, fit a high-capacity per-video model to predict its RSD label.
4. Compute per-frame absolute residuals on that same training video.
5. Keep frames with unusually small residuals for that video.
6. Train the general model on selected training frames.
7. Evaluate on untouched validation/test videos.

This is data selection, not final evaluation. The validation/test sets must never influence which frames are kept.

## 0. What This Notebook Does Not Do

The `2019/` folder includes copyrighted/proprietary code and artifacts. This notebook does not import or copy those modules. It reimplements the method at the algorithm level using fresh, minimal code.

Publication-safe framing:

> Controlled per-video overfitting is used as a training-set-only label-quality diagnostic. Frames that remain hard to fit even when the model is allowed to memorize a single training video are treated as lower-confidence labels. The final model is trained on the high-confidence subset and evaluated on untouched validation/test videos.

Reviewer-sensitive point: selecting frames from validation or test videos would be leakage. The functions below explicitly audit for that.

In [ ]:
# Setup
from pathlib import Path
from collections import Counter
import json
import math
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (11, 4.5)

PROJECT = Path.cwd()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent

MANIFEST_CANDIDATES = [
    PROJECT / 'lambda_mirror/labels/mb140_fold0_labels_kmeans.json',
    PROJECT / 'labels/mb140_fold0_labels_kmeans.json',
    Path('/lambda/nfs/bariatric-rsd/labels/mb140_fold0_labels_kmeans.json'),
]

DATA_ROOT_CANDIDATES = [
    PROJECT,
    PROJECT / 'lambda_mirror',
    PROJECT / 'data',
    Path('/lambda/nfs/bariatric-rsd/extern/MultiBypass140/datasets/MultiBypass140'),
]

LABELS = next((p for p in MANIFEST_CANDIDATES if p.exists()), None)
FIG_DIR = PROJECT / 'notebooks' / 'overfit_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT)
print('Manifest:', LABELS)
print('Figure directory:', FIG_DIR)

In [ ]:
# Load current manifest and build video/frame tables.
if LABELS is None:
    raise FileNotFoundError('Could not find mb140_fold0_labels_kmeans.json. Update MANIFEST_CANDIDATES.')

videos = json.loads(LABELS.read_text())
print(f'Loaded {len(videos)} videos from {LABELS}')

def build_tables(videos):
    video_rows = []
    frame_rows = []
    for video in videos:
        vid = str(video['video_id'])
        frames = video.get('frames', [])
        duration_sec = float(video.get('total_duration_sec', np.nan))
        video_rows.append({
            'video_id': vid,
            'split': video.get('split', 'train'),
            'phase_order_cluster': video.get('phase_order_cluster', np.nan),
            'duration_min': duration_sec / 60.0,
            'n_frames': len(frames),
            'n_deviation_frames': int(sum(bool(f.get('is_deviation', False)) for f in frames)),
            'phase_count': len(video.get('phase_sequence', [])),
            'phase_sequence': ' -> '.join(video.get('phase_sequence', [])),
        })
        for j, frame in enumerate(frames):
            frame_rows.append({
                'video_id': vid,
                'split': video.get('split', 'train'),
                'row_in_video': j,
                'frame_idx': frame.get('frame_idx'),
                'timestamp_sec': float(frame.get('timestamp_sec', np.nan)),
                'elapsed_min': float(frame.get('timestamp_sec', np.nan)) / 60.0,
                'rsd_sec': float(frame.get('rsd_sec', np.nan)),
                'rsd_min': float(frame.get('rsd_sec', np.nan)) / 60.0,
                'rsd_normalized': float(frame.get('rsd_normalized', np.nan)),
                'phase': frame.get('phase'),
                'is_deviation': bool(frame.get('is_deviation', False)),
                'frame_path': str(frame.get('frame_path')),
            })
    return pd.DataFrame(video_rows), pd.DataFrame(frame_rows)

video_df, frame_df = build_tables(videos)

display(video_df.groupby('split').agg(videos=('video_id', 'count'), frames=('n_frames', 'sum')).reset_index())
display(video_df.groupby(['split', 'phase_order_cluster']).size().unstack(fill_value=0))
display(video_df.head())

## 1. Leakage-Safe Protocol

The selection model is allowed to overfit each training video because its output is not used as the final model. It is only a label-quality diagnostic.

The critical rule is simple: residuals used for selection must come only from training videos. Validation and test videos remain untouched so the final comparison is honest.

In [ ]:
KEY_COLS = ['video_id', 'frame_path']

def split_audit_for_selection(candidate_df, frame_df, video_col='video_id', frame_col='frame_path'):
    """Return a table showing whether a candidate residual/selection table touches non-train frames."""
    probe = candidate_df[[video_col, frame_col]].copy()
    probe[video_col] = probe[video_col].astype(str)
    probe[frame_col] = probe[frame_col].astype(str)

    ref = frame_df[['video_id', 'frame_path', 'split']].copy()
    ref['video_id'] = ref['video_id'].astype(str)
    ref['frame_path'] = ref['frame_path'].astype(str)

    merged = probe.merge(ref, left_on=[video_col, frame_col], right_on=['video_id', 'frame_path'], how='left')
    audit = merged['split'].fillna('missing').value_counts().rename_axis('split').reset_index(name='rows')
    return audit


def assert_train_only(candidate_df, frame_df, video_col='video_id', frame_col='frame_path'):
    audit = split_audit_for_selection(candidate_df, frame_df, video_col, frame_col)
    bad = audit[~audit['split'].isin(['train'])]
    if len(bad):
        raise ValueError('Selection inputs include non-train or missing frames:\n' + bad.to_string(index=False))
    return audit

# The full frame table includes val/test, so this should fail if used directly.
try:
    assert_train_only(frame_df, frame_df)
except ValueError as e:
    print('Expected guardrail triggered: full manifest is not train-only.')
    print(str(e).split('\n')[0])

## 2. Selection Rule

The historical filter kept frames whose per-video overfit residual was below a small multiple of that video's residual scale.

Default rule in this notebook:

`keep(frame i in video v) = abs_residual_i < k * std(abs_residual_v)`

The 2019 threshold was approximately `k = 0.385`. That is intentionally aggressive. For the modern paper, treat `k` as an ablation parameter rather than a fixed truth. Recommended sweep: `0.385`, `0.75`, `1.0`, `1.5`, `2.0`.

In [ ]:
def select_by_overfit_residuals(
    residual_df,
    k=0.385,
    video_col='video_id',
    frame_col='frame_path',
    residual_col='abs_residual',
    min_keep_frac=None,
    fallback_quantile=None,
):
    """
    Select high-confidence frames from per-video overfit residuals.

    Parameters
    ----------
    residual_df:
        DataFrame with at least video_id, frame_path, abs_residual.
    k:
        Threshold multiplier. Historical value: 0.385.
    min_keep_frac:
        Optional guardrail. If a video keeps less than this fraction, relax using fallback_quantile.
    fallback_quantile:
        Optional quantile threshold used only when min_keep_frac is violated.

    Returns
    -------
    selected_df, stats_df
    """
    required = {video_col, frame_col, residual_col}
    missing = required - set(residual_df.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')

    selected_parts = []
    stat_rows = []

    for vid, group in residual_df.groupby(video_col, sort=False):
        group = group.copy()
        resid = group[residual_col].astype(float).to_numpy()
        scale = float(np.std(resid, ddof=0))
        threshold = k * scale

        if not np.isfinite(threshold) or threshold <= 0:
            keep = np.ones(len(group), dtype=bool)
            reason = 'degenerate_scale_keep_all'
        else:
            keep = resid < threshold
            reason = 'std_threshold'

        if min_keep_frac is not None and keep.mean() < min_keep_frac:
            if fallback_quantile is None:
                raise ValueError(f'{vid}: kept {keep.mean():.3f}, below min_keep_frac={min_keep_frac}')
            threshold = float(np.quantile(resid, fallback_quantile))
            keep = resid <= threshold
            reason = f'fallback_quantile_{fallback_quantile}'

        kept = group.loc[keep].copy()
        kept['selection_threshold'] = threshold
        kept['selection_k'] = k
        kept['selection_reason'] = reason
        selected_parts.append(kept)

        stat_rows.append({
            'video_id': str(vid),
            'n_frames': len(group),
            'n_kept': int(keep.sum()),
            'kept_frac': float(keep.mean()),
            'residual_mean': float(np.mean(resid)),
            'residual_std': scale,
            'threshold': threshold,
            'reason': reason,
        })

    selected_df = pd.concat(selected_parts, ignore_index=True) if selected_parts else residual_df.iloc[0:0].copy()
    stats_df = pd.DataFrame(stat_rows)
    return selected_df, stats_df


def standardize_residual_table(df):
    """Normalize common residual CSV schemas into video_id, frame_path, prediction, target, abs_residual."""
    df = df.copy()
    rename = {}
    aliases = {
        'video_name': 'video_id',
        'vid': 'video_id',
        'img_path': 'frame_path',
        'frame_name': 'frame_path',
        'actual': 'target',
        'label': 'target',
        'rsd_normalized': 'target',
        'pred': 'prediction',
    }
    for old, new in aliases.items():
        if old in df.columns and new not in df.columns:
            rename[old] = new
    df = df.rename(columns=rename)

    required_base = {'video_id', 'frame_path'}
    missing = required_base - set(df.columns)
    if missing:
        raise ValueError(f'Residual table must include {sorted(required_base)}; missing {sorted(missing)}')

    if 'abs_residual' not in df.columns:
        if {'prediction', 'target'} <= set(df.columns):
            df['abs_residual'] = (df['prediction'].astype(float) - df['target'].astype(float)).abs()
        else:
            raise ValueError('Need abs_residual or both prediction and target columns.')

    df['video_id'] = df['video_id'].astype(str)
    df['frame_path'] = df['frame_path'].astype(str)
    df['abs_residual'] = df['abs_residual'].astype(float)
    return df

## 3. Fast Local Demo Without Images

The real selection must use residuals from a per-video overfit model. This local demo creates a synthetic residual proxy from known weak regions: video endpoints, phase transitions, and deviation frames. It is only for checking the mechanics, plots, and manifest writing code.

Do not report the synthetic demo as experimental evidence.

In [ ]:
def make_synthetic_residuals(frame_df, max_train_videos=None, seed=42):
    rng = np.random.default_rng(seed)
    train_videos = sorted(frame_df.loc[frame_df['split'] == 'train', 'video_id'].unique())
    if max_train_videos is not None:
        train_videos = train_videos[:max_train_videos]
    rows = []

    for vid in train_videos:
        g = frame_df[frame_df['video_id'] == vid].sort_values('timestamp_sec').reset_index(drop=True)
        n = len(g)
        if n == 0:
            continue
        progress = np.linspace(0, 1, n)

        # Endpoint frames and phase transitions are plausible label-quality stress points.
        endpoint_penalty = 0.015 * (np.exp(-progress / 0.08) + np.exp(-(1 - progress) / 0.08))
        phase_change = g['phase'].ne(g['phase'].shift(1)).astype(float).to_numpy()
        transition_penalty = np.convolve(phase_change, np.ones(31) / 31, mode='same') * 0.02
        deviation_penalty = g['is_deviation'].astype(float).to_numpy() * 0.01
        noise = np.abs(rng.normal(0, 0.006, n))

        abs_residual = endpoint_penalty + transition_penalty + deviation_penalty + noise
        for frame_path, resid in zip(g['frame_path'], abs_residual):
            rows.append({
                'video_id': vid,
                'frame_path': frame_path,
                'abs_residual': float(resid),
                'source': 'synthetic_demo_not_evidence',
            })
    return pd.DataFrame(rows)

demo_residual_df = make_synthetic_residuals(frame_df, max_train_videos=None)
print(f'Demo residual rows: {len(demo_residual_df):,}')
display(assert_train_only(demo_residual_df, frame_df))

demo_selected, demo_stats = select_by_overfit_residuals(demo_residual_df, k=0.385)
display(demo_stats.describe(include='all'))
display(demo_stats.sort_values('kept_frac').head())

In [ ]:
def plot_selection_for_video(residual_df, selected_df, video_id, frame_df, save_path=None):
    g = residual_df[residual_df['video_id'].astype(str) == str(video_id)].copy()
    if g.empty:
        raise ValueError(f'No residual rows for {video_id}')

    ref = frame_df[['video_id', 'frame_path', 'timestamp_sec', 'phase', 'is_deviation']].copy()
    g = g.merge(ref, on=['video_id', 'frame_path'], how='left')

    selected_keys = set(zip(selected_df['video_id'].astype(str), selected_df['frame_path'].astype(str)))
    g['kept'] = [(str(v), str(p)) in selected_keys for v, p in zip(g['video_id'], g['frame_path'])]
    g = g.sort_values('timestamp_sec')

    threshold = float(g.loc[g['kept'], 'selection_threshold'].iloc[0]) if g['kept'].any() and 'selection_threshold' in g else np.nan

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
    x = g['timestamp_sec'].to_numpy() / 60.0
    axes[0].scatter(x[~g['kept'].to_numpy()], g.loc[~g['kept'], 'abs_residual'], s=6, color='C3', alpha=0.55, label='dropped')
    axes[0].scatter(x[g['kept'].to_numpy()], g.loc[g['kept'], 'abs_residual'], s=6, color='C2', alpha=0.75, label='kept')
    if np.isfinite(threshold):
        axes[0].axhline(threshold, color='black', linestyle='--', linewidth=1, label='threshold')
    axes[0].set_title(f'{video_id}: residuals over time')
    axes[0].set_xlabel('elapsed time (min)')
    axes[0].set_ylabel('absolute residual')
    axes[0].legend()
    axes[0].grid(alpha=0.25)

    axes[1].hist(g['abs_residual'], bins=60, color='C0', alpha=0.8)
    if np.isfinite(threshold):
        axes[1].axvline(threshold, color='black', linestyle='--', linewidth=1)
    axes[1].set_title(f'{video_id}: residual histogram')
    axes[1].set_xlabel('absolute residual')
    axes[1].set_ylabel('frames')
    axes[1].grid(alpha=0.25)

    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=180, bbox_inches='tight')
    return fig

for vid in demo_stats.sort_values('kept_frac')['video_id'].head(3):
    plot_selection_for_video(demo_residual_df, demo_selected, vid, frame_df, FIG_DIR / f'synthetic_selection_{vid}.png')
plt.show()

In [ ]:
def plot_threshold_sweep(residual_df, ks=(0.385, 0.75, 1.0, 1.5, 2.0), save_path=None):
    rows = []
    for k in ks:
        _, stats = select_by_overfit_residuals(residual_df, k=k)
        rows.append({
            'k': k,
            'mean_kept_frac': stats['kept_frac'].mean(),
            'median_kept_frac': stats['kept_frac'].median(),
            'min_kept_frac': stats['kept_frac'].min(),
            'total_kept_frames': stats['n_kept'].sum(),
        })
    sweep = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(sweep['k'], sweep['mean_kept_frac'], marker='o', label='mean video kept fraction')
    ax.plot(sweep['k'], sweep['median_kept_frac'], marker='o', label='median video kept fraction')
    ax.axvline(0.385, color='C3', linestyle='--', label='2019 threshold')
    ax.set_xlabel('k in threshold = k * std(abs residual)')
    ax.set_ylabel('kept fraction')
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=180, bbox_inches='tight')
    return sweep, fig

sweep_df, _ = plot_threshold_sweep(demo_residual_df, save_path=FIG_DIR / 'synthetic_threshold_sweep.png')
display(sweep_df)
plt.show()

## 4. Real Mode A: Apply an Existing Overfit Residual CSV

Use this if you have already run per-video overfit models and saved predictions.

Expected columns, with aliases accepted:

- `video_id` or `video_name`
- `frame_path` or `img_path` or `frame_name`
- either `abs_residual`, or both `prediction` and `target`/`actual`/`label`

The CSV must contain training videos only. The audit below enforces that.

In [ ]:
RESIDUAL_CSV = PROJECT / 'outputs' / 'overfit_residuals.csv'
K_FOR_SELECTION = 0.385

if RESIDUAL_CSV.exists():
    residual_df = standardize_residual_table(pd.read_csv(RESIDUAL_CSV))
    display(assert_train_only(residual_df, frame_df))
    selected_df, selection_stats_df = select_by_overfit_residuals(residual_df, k=K_FOR_SELECTION)
    display(selection_stats_df.describe())
else:
    print(f'No residual CSV found at {RESIDUAL_CSV}')
    print('Set RESIDUAL_CSV to your per-video overfit prediction CSV, then rerun this cell.')

## 5. Real Mode B: Generate Residuals From Cached Features

This mode avoids decoding images in the notebook. It expects a feature cache with one row per frame.

Recommended cache format: `.npz` with arrays:

- `features`: shape `[N, D]`
- `video_id`: shape `[N]`
- `frame_path`: shape `[N]`
- `target`: shape `[N]`, normalized RSD target

The per-video selector below fits a small MLP on cached features for one video at a time. It intentionally evaluates on the same video because the goal is label-quality selection, not model validation.

In [ ]:
def fit_per_video_mlp_residuals(features, targets, epochs=250, lr=1e-3, hidden=256, seed=42):
    import torch
    import torch.nn as nn

    torch.manual_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    x = torch.as_tensor(features, dtype=torch.float32, device=device)
    y = torch.as_tensor(targets, dtype=torch.float32, device=device)

    model = nn.Sequential(
        nn.Linear(x.shape[1], hidden), nn.GELU(),
        nn.Linear(hidden, hidden), nn.GELU(),
        nn.Linear(hidden, 1), nn.Sigmoid(),
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    loss_fn = nn.MSELoss()
    best_loss = float('inf')
    patience = 30
    stale = 0

    for epoch in range(epochs):
        opt.zero_grad(set_to_none=True)
        pred = model(x).squeeze(-1)
        loss = loss_fn(pred, y)
        loss.backward()
        opt.step()

        loss_value = float(loss.detach().cpu())
        if loss_value + 1e-8 < best_loss:
            best_loss = loss_value
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    with torch.no_grad():
        pred = model(x).squeeze(-1).detach().cpu().numpy()
    residuals = np.abs(pred - np.asarray(targets, dtype=float))
    return pred, residuals, best_loss


def residuals_from_feature_cache(feature_npz, max_videos=None):
    data = np.load(feature_npz, allow_pickle=True)
    features = data['features']
    video_ids = data['video_id'].astype(str)
    frame_paths = data['frame_path'].astype(str)
    targets = data['target'].astype(float)

    rows = []
    unique_videos = list(dict.fromkeys(video_ids))
    if max_videos is not None:
        unique_videos = unique_videos[:max_videos]

    for vid in unique_videos:
        idx = np.where(video_ids == vid)[0]
        pred, resid, loss = fit_per_video_mlp_residuals(features[idx], targets[idx])
        for p, yhat, y, r in zip(frame_paths[idx], pred, targets[idx], resid):
            rows.append({
                'video_id': vid,
                'frame_path': p,
                'prediction': float(yhat),
                'target': float(y),
                'abs_residual': float(r),
                'selector_loss': float(loss),
            })
        print(f'{vid}: frames={len(idx):,}, final selector MSE={loss:.6f}')
    return pd.DataFrame(rows)

FEATURE_CACHE = PROJECT / 'outputs' / 'frame_features_train.npz'
if FEATURE_CACHE.exists():
    residual_df = residuals_from_feature_cache(FEATURE_CACHE)
    display(assert_train_only(residual_df, frame_df))
    selected_df, selection_stats_df = select_by_overfit_residuals(residual_df, k=K_FOR_SELECTION)
    display(selection_stats_df.describe())
else:
    print(f'No feature cache found at {FEATURE_CACHE}')

## 6. Real Mode C: Optional Frame Encoding Demo

This mode directly loads frame images, encodes them, and overfits a per-video head. It is off by default because it requires local media and `timm`.

Use it for a small pilot only. For the full dataset, prefer generating a feature cache once and using Real Mode B.

In [ ]:
RUN_FRAME_DEMO = False
PILOT_VIDEO_ID = None  # e.g. 'BBP01'
MAX_FRAMES_FOR_DEMO = 1200


def resolve_frame_path(relative_path):
    rel = Path(str(relative_path))
    for root in DATA_ROOT_CANDIDATES:
        candidate = root / rel
        if candidate.exists():
            return candidate
    return None


def encode_frames_with_timm(frame_paths, model_name='vit_base_patch16_224', batch_size=64):
    import torch
    import timm
    from PIL import Image
    import torchvision.transforms as T

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    encoder = timm.create_model(model_name, pretrained=True, num_classes=0, dynamic_img_size=True)
    encoder.eval().to(device)
    for p in encoder.parameters():
        p.requires_grad = False

    chunks = []
    with torch.no_grad():
        for start in range(0, len(frame_paths), batch_size):
            batch_paths = frame_paths[start:start + batch_size]
            images = [transform(Image.open(p).convert('RGB')) for p in batch_paths]
            x = torch.stack(images).to(device)
            chunks.append(encoder(x).detach().cpu().numpy())
    return np.concatenate(chunks, axis=0)

if RUN_FRAME_DEMO:
    train_vids = video_df.loc[video_df['split'] == 'train', 'video_id'].tolist()
    vid = PILOT_VIDEO_ID or train_vids[0]
    g = frame_df[frame_df['video_id'] == vid].sort_values('timestamp_sec').head(MAX_FRAMES_FOR_DEMO)
    resolved = [resolve_frame_path(p) for p in g['frame_path']]
    missing = sum(p is None for p in resolved)
    if missing:
        raise FileNotFoundError(f'{missing}/{len(resolved)} frame paths were not found. Update DATA_ROOT_CANDIDATES.')

    features = encode_frames_with_timm(resolved)
    pred, resid, loss = fit_per_video_mlp_residuals(features, g['rsd_normalized'].to_numpy())
    frame_demo_residual_df = pd.DataFrame({
        'video_id': vid,
        'frame_path': g['frame_path'].to_numpy(),
        'prediction': pred,
        'target': g['rsd_normalized'].to_numpy(),
        'abs_residual': resid,
    })
    frame_demo_selected, frame_demo_stats = select_by_overfit_residuals(frame_demo_residual_df, k=K_FOR_SELECTION)
    display(frame_demo_stats)
    plot_selection_for_video(frame_demo_residual_df, frame_demo_selected, vid, frame_df)
else:
    print('RUN_FRAME_DEMO is False. Set it to True only where frames and timm are available.')

## 7. Write a Filtered Training Manifest

This function writes a new label JSON that filters only training frames. Validation and test videos are copied unchanged.

Important consequence for the current `BariatricFrameDataset`: filtering too aggressively can reduce the number of valid temporal windows. The summary table estimates this before writing.

In [ ]:
def estimate_sequence_samples(n_frames, sequence_len=8, frame_stride=5):
    return max(0, len(range(0, int(n_frames) - sequence_len * frame_stride, frame_stride)))


def summarize_manifest_after_selection(videos, selected_df, sequence_len=8, frame_stride=5):
    selected_keys = set(zip(selected_df['video_id'].astype(str), selected_df['frame_path'].astype(str)))
    rows = []
    for video in videos:
        vid = str(video['video_id'])
        split = video.get('split', 'train')
        old_n = len(video.get('frames', []))
        if split == 'train':
            new_n = sum((vid, str(f.get('frame_path'))) in selected_keys for f in video.get('frames', []))
        else:
            new_n = old_n
        rows.append({
            'video_id': vid,
            'split': split,
            'old_frames': old_n,
            'new_frames': new_n,
            'kept_frac': new_n / old_n if old_n else np.nan,
            'old_samples_est': estimate_sequence_samples(old_n, sequence_len, frame_stride),
            'new_samples_est': estimate_sequence_samples(new_n, sequence_len, frame_stride),
        })
    return pd.DataFrame(rows)


def assert_all_train_videos_have_selection(videos, selected_df):
    train_ids = {str(v['video_id']) for v in videos if v.get('split', 'train') == 'train'}
    selected_ids = set(selected_df['video_id'].astype(str))
    missing = sorted(train_ids - selected_ids)
    if missing:
        preview = ', '.join(missing[:10])
        raise ValueError(f'Selection is missing {len(missing)} training videos. First missing: {preview}')
    return True


def write_filtered_manifest(videos, selected_df, output_path, k, notes='controlled_overfit_data_selection'):
    assert_all_train_videos_have_selection(videos, selected_df)
    selected_keys = set(zip(selected_df['video_id'].astype(str), selected_df['frame_path'].astype(str)))
    output = []
    for video in videos:
        new_video = {key: value for key, value in video.items() if key != 'frames'}
        vid = str(video['video_id'])
        split = video.get('split', 'train')
        frames = video.get('frames', [])
        if split == 'train':
            kept_frames = [f for f in frames if (vid, str(f.get('frame_path'))) in selected_keys]
            new_video['frames'] = kept_frames
            new_video['selection_method'] = {
                'name': 'controlled_overfit_residual_filter',
                'k': float(k),
                'input_split': 'train_only',
                'old_frame_count': len(frames),
                'new_frame_count': len(kept_frames),
                'notes': notes,
            }
        else:
            new_video['frames'] = frames
            new_video['selection_method'] = {
                'name': 'not_filtered_eval_split',
                'input_split': split,
                'old_frame_count': len(frames),
                'new_frame_count': len(frames),
            }
        output.append(new_video)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(output, indent=2))
    return output_path

# Demo summary using synthetic residuals. Replace demo_selected with selected_df from Real Mode A/B/C.
demo_manifest_summary = summarize_manifest_after_selection(videos, demo_selected)
display(demo_manifest_summary.groupby('split').agg(
    videos=('video_id', 'count'),
    old_frames=('old_frames', 'sum'),
    new_frames=('new_frames', 'sum'),
    old_samples_est=('old_samples_est', 'sum'),
    new_samples_est=('new_samples_est', 'sum'),
    median_kept_frac=('kept_frac', 'median'),
).round(3))

In [ ]:
WRITE_DEMO_MANIFEST = False
OUTPUT_MANIFEST = PROJECT / 'lambda_mirror/labels/mb140_fold0_labels_kmeans_overfit_selected.json'

if WRITE_DEMO_MANIFEST:
    # Replace demo_selected with selected_df from a real residual table before using this for training.
    path = write_filtered_manifest(videos, demo_selected, OUTPUT_MANIFEST, k=0.385, notes='synthetic_demo_do_not_train')
    print('Wrote:', path)
else:
    print('WRITE_DEMO_MANIFEST is False.')
    print('After Real Mode A/B/C, run:')
    print("write_filtered_manifest(videos, selected_df, OUTPUT_MANIFEST, k=K_FOR_SELECTION)")

## 8. Recommended Experiment Plan

Run the following as paired experiments against the current best RSD model.

1. Generate overfit residuals from training videos only.
2. Produce filtered manifests for `k in [0.385, 0.75, 1.0, 1.5, 2.0]`.
3. Train the same model/config/seeds on each filtered manifest.
4. Evaluate on the same unfiltered validation/test videos.
5. Report video-level paired bootstrap confidence intervals for MAE difference.
6. Plot kept-frame fraction versus validation/test MAE.

Reviewer-facing ablations:

- no filtering baseline
- random frame subsampling matched to the same kept-frame count
- controlled overfit filtering
- relaxed threshold sweep
- optional robust residual scale, e.g. MAD, as a sensitivity analysis

## 9. Draft Method Text

**Controlled overfitting frame selection.** Surgical video labels contain noisy regions, especially near setup, out-of-body periods, phase boundaries, and annotation irregularities. We therefore use a training-set-only frame selection procedure inspired by our 2019 pilot work. For each training video independently, we train a small selector model to overfit the normalized RSD trajectory of that video. The selector is then evaluated on the same video, producing an absolute residual for every frame. Frames with residual below `k` times the within-video residual standard deviation are retained as high-confidence training frames. The procedure is never applied to validation or test videos, and selector predictions are not used at inference. The final RSD model is trained from scratch on the selected training frames and evaluated on untouched held-out videos.

**Leakage prevention.** Video splits are fixed before selection. The selector sees only training videos, and all threshold statistics are computed within each training video. Validation and test frames are copied unchanged into the manifest only for evaluation. We report results against both the unfiltered baseline and a random-subsampling control with the same number of retained training frames.

## 10. Interpretation Rules

This approach succeeds only if it improves held-out validation/test performance over the unfiltered baseline and a matched random-subsampling control.

Strong positive result:

- filtered training improves RSD MAE
- random matched subsampling does not
- gains are largest near endpoints/phase transitions/noisy videos
- calibration does not get worse

Negative result:

- filtered training performs like random subsampling, or worse
- too many temporal windows disappear after filtering
- the selector removes clinically important but rare frames

If the negative result happens, the paper can still use the 2019 idea as motivation for data-quality analysis, but it should not become the main contribution.